In [ ]:
from google.colab import files
uploaded = files.upload()

Saving scenario_model_v3.graphml to scenario_model_v3 (1).graphml


In [ ]:
from google.colab import files

# This will prompt you to pick a file from your Mac.
# Select your scenario_model.graphml (or whatever it’s named).
uploaded = files.upload()


Saving scenario_model_v3.graphml to scenario_model_v3 (2).graphml


In [ ]:
# List files in the working directory
!ls -lh


total 108K
-rw-r--r-- 1 root root 3.3K Jun 28 03:54  risk_mat.npy
drwxr-xr-x 1 root root 4.0K Jun 26 13:35  sample_data
-rw-r--r-- 1 root root  13K Jun 28 03:48  scenario_model.graphml
-rw-r--r-- 1 root root  20K Jun 28 03:49  scenario_model_v2.graphml
-rw-r--r-- 1 root root  20K Jun 28 03:55 'scenario_model_v3 (1).graphml'
-rw-r--r-- 1 root root  20K Jun 28 03:56 'scenario_model_v3 (2).graphml'
-rw-r--r-- 1 root root  20K Jun 28 03:52  scenario_model_v3.graphml
-rw-r--r-- 1 root root 3.3K Jun 28 03:54  trust_mat.npy


In [ ]:
# Rename your primary GraphML file
!mv "scenario_model (2).graphml" scenario_model.graphml

# Verify it’s been renamed
!ls -lh


mv: cannot stat 'scenario_model (2).graphml': No such file or directory
total 108K
-rw-r--r-- 1 root root 3.3K Jun 28 03:54  risk_mat.npy
drwxr-xr-x 1 root root 4.0K Jun 26 13:35  sample_data
-rw-r--r-- 1 root root  13K Jun 28 03:48  scenario_model.graphml
-rw-r--r-- 1 root root  20K Jun 28 03:49  scenario_model_v2.graphml
-rw-r--r-- 1 root root  20K Jun 28 03:55 'scenario_model_v3 (1).graphml'
-rw-r--r-- 1 root root  20K Jun 28 03:56 'scenario_model_v3 (2).graphml'
-rw-r--r-- 1 root root  20K Jun 28 03:52  scenario_model_v3.graphml
-rw-r--r-- 1 root root 3.3K Jun 28 03:54  trust_mat.npy


In [ ]:
%%bash
sed -i '2i\
<key id="ktrust"   for="edge" attr.name="trustScore"   attr.type="double"/>;\
<key id="krisk"    for="edge" attr.name="riskWeight"    attr.type="double"/>;\
<key id="urgency"  for="node" attr.name="urgency"       attr.type="double"/>;\
<key id="health"   for="node" attr.name="healthStatus"  attr.type="string"/>;\
<key id="cyber"    for="node" attr.name="cyberEvent"    attr.type="string"/>' \
scenario_model.graphml


In [ ]:
!head -n 20 scenario_model_v2.graphml


<?xml version='1.0' encoding='utf-8'?>
<ns0:graphml xmlns:ns0="http://graphml.graphdrawing.org/xmlns" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://graphml.graphdrawing.org/xmlns http://graphml.graphdrawing.org/xmlns/1.0/graphml.xsd"><ns0:key id="d21" for="edge" attr.name="isBlocked" attr.type="boolean" />
<ns0:key id="d20" for="edge" attr.name="eta" attr.type="string" />
<ns0:key id="d19" for="edge" attr.name="assignedAt" attr.type="string" />
<ns0:key id="d18" for="edge" attr.name="requestedAt" attr.type="string" />
<ns0:key id="d17" for="edge" attr.name="requestedAt" attr.type="double" />
<ns0:key id="d16" for="edge" attr.name="distance" attr.type="double" />
<ns0:key id="d15" for="edge" attr.name="isBlocked" attr.type="double" />
<ns0:key id="d14" for="edge" attr.name="eta" attr.type="double" />
<ns0:key id="d13" for="edge" attr.name="assignedAt" attr.type="double" />
<ns0:key id="d12" for="edge" attr.name="type" attr.type="string" />
<ns0:key id=

In [ ]:
from google.colab import files

# Upload from your Mac
uploaded = files.upload()
# After selecting your file(s), they’ll appear in /content/


TypeError: 'NoneType' object is not subscriptable

In [ ]:
# List what you just uploaded
!ls -lh

# Rename whichever file is your scenario model.
# Adjust the exact filename if it’s slightly different.
!mv "scenario_model (2).graphml" scenario_model.graphml

# Confirm the rename
!ls -lh


In [ ]:
import xml.etree.ElementTree as ET

# Parse the uploaded file
tree = ET.parse('scenario_model.graphml')
root = tree.getroot()

# Namespaces handling
ns_uri = root.tag.split('}')[0].strip('{')
ns     = {'g': ns_uri}

# Default values to add
node_defaults = {
    'urgency':      '0.5',
    'healthStatus': 'stable',
    'cyberEvent':   ''
}
edge_defaults = {
    'trustScore':  '0.8',
    'riskWeight':  '1.2'
}

# Add missing <data> for nodes
for node in root.findall('.//g:node', ns):
    existing = {d.attrib['key'] for d in node.findall('g:data', ns)}
    for key, val in node_defaults.items():
        if key not in existing:
            d = ET.SubElement(node, f'{{{ns_uri}}}data', key=key)
            d.text = val

# Add missing <data> for edges
for edge in root.findall('.//g:edge', ns):
    existing = {d.attrib['key'] for d in edge.findall('g:data', ns)}
    for key, val in edge_defaults.items():
        if key not in existing:
            d = ET.SubElement(edge, f'{{{ns_uri}}}data', key=key)
            d.text = val

# Write out the enriched version
tree.write('scenario_model_v2.graphml', encoding='utf-8', xml_declaration=True)
print("Created scenario_model_v2.graphml")


In [ ]:
# Show it’s there
!ls -lh

# Peek at the first 30 lines to confirm your <data> tags appear
!head -n 30 scenario_model_v2.graphml


In [ ]:
import xml.etree.ElementTree as ET

# 1. Parse the enriched file you already created
tree = ET.parse('scenario_model_v2.graphml')
root = tree.getroot()

# 2. Grab the GraphML namespace URI
ns_uri = root.tag.split('}')[0].strip('{')
gkey = f'{{{ns_uri}}}key'
ggraph = f'{{{ns_uri}}}graph'

# 3. Prepare the key definitions you need
new_keys = [
    # (id, for, attr.name, attr.type)
    ('urgency',     'node', 'urgency',     'double'),
    ('healthStatus','node', 'healthStatus','string'),
    ('cyberEvent',  'node', 'cyberEvent',  'string'),
    ('trustScore',  'edge', 'trustScore',  'double'),
    ('riskWeight',  'edge', 'riskWeight',  'double'),
]

# 4. Find the index of the <graph> element so we insert keys before it
children = list(root)
insert_at = next(i for i, el in enumerate(children) if el.tag == ggraph)

# 5. Insert each <key> element
for key_id, key_for, attr_name, attr_type in new_keys:
    elem = ET.Element(gkey, id=key_id, **{
        'for': key_for,
        'attr.name': attr_name,
        'attr.type': attr_type
    })
    root.insert(insert_at, elem)
    insert_at += 1

# 6. Write out a new file
tree.write('scenario_model_v3.graphml', encoding='utf-8', xml_declaration=True)
print("✅ Created scenario_model_v3.graphml with your new key definitions.")


In [ ]:
!ls -lh
!head -n 20 scenario_model_v3.graphml


In [ ]:
import networkx as nx
G = nx.read_graphml('scenario_model_v3.graphml')
print("Loaded successfully! Nodes:", len(G.nodes()), "Edges:", len(G.edges()))


In [ ]:
# 1A. List files to confirm scenario_model_v3.graphml is present
!ls -lh

# 1B. Peek at the first 20 lines to see your <key> definitions
!head -n 20 scenario_model_v3.graphml


In [ ]:
import networkx as nx

# 2A. Load the enriched GraphML
G = nx.read_graphml('scenario_model_v3.graphml')

# 2B. Quick sanity check
print(f"✅ Loaded graph with {len(G.nodes())} nodes and {len(G.edges())} edges.")
# Optionally print a sample node & edge to confirm data keys exist
sample_node = next(iter(G.nodes(data=True)))
sample_edge = next(iter(G.edges(data=True)))
print("Sample node data:", sample_node)
print("Sample edge data:", sample_edge)


In [ ]:
import numpy as np

nodes = list(G.nodes())
n = len(nodes)
risk  = np.zeros((n, n))
trust = np.zeros((n, n))

for i, u in enumerate(nodes):
    for j, v in enumerate(nodes):
        edge_data = G.get_edge_data(u, v)
        if edge_data is None:
            # no edge here
            continue

        # If it's a flat dict with our keys, use it directly
        if 'riskWeight' in edge_data and 'trustScore' in edge_data:
            attrs = edge_data
        else:
            # Otherwise assume it's nested: take the first sub-dict
            # (e.g. networkx stored a dict of dicts)
            first_key = next(iter(edge_data))
            attrs = edge_data[first_key]

        risk[i, j]  = float(attrs['riskWeight'])
        trust[i, j] = float(attrs['trustScore'])

# Save them
np.save('risk_mat.npy',  risk)
np.save('trust_mat.npy', trust)

print("✅ Export complete.")
print("risk_mat.npy:", risk.shape)
print("trust_mat.npy:", trust.shape)


In [ ]:
import numpy as np
print(np.load('risk_mat.npy').shape, np.load('trust_mat.npy').shape)
